# 🔗 BigQuery Property Graph Auto-Builder

This notebook automatically converts your existing BigQuery tables into a **Property Graph** — no manual schema design required.

It uses **Gemini** to analyze your table schemas, identify the right nodes and edges, generate the graph DDL, and then produce a set of sample GQL questions you can ask immediately.

### What you need
- A BigQuery dataset with relational tables
- A short description of your domain
- A few examples of the business questions you want to answer

### What you get
- ✅ Automatic node and edge identification with reasoning
- ✅ A `CREATE PROPERTY GRAPH` DDL statement, ready to run
- ✅ 5 sample GQL queries tailored to your graph and business questions

---

## Flow

```
BigQuery Tables
      │
      ▼
Step 1: Extract Schema (INFORMATION_SCHEMA + sample rows)
      │
      ▼
Step 2: Gemini identifies Nodes & Edges
      │
      ▼
Step 3: Generate & Review CREATE PROPERTY GRAPH DDL
      │
      ▼
Step 4: Create the Property Graph in BigQuery
      │
      ▼
Step 5: Auto-generate Sample GQL Questions & Queries
      │
      ▼
Step 6: Run your first query
```

## 🛠️ Setup

In [ ]:
!pip install google-cloud-bigquery google-cloud-aiplatform vertexai --quiet

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown
from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

## ⚙️ Configuration

This is the only section you need to edit.

**Tips for writing good business questions:**
- Be specific about the entities involved (customers, products, stores)
- Include multi-hop questions ("customers who bought X and visited Y")
- Include aggregation questions ("most popular", "top 10", "how many")
- The more questions you provide, the better Gemini can design the graph

In [ ]:
# ── Project & Dataset ────────────────────────────────────────
PROJECT_ID   = "your-project-id"       # @param {type:"string"}
DATASET_ID   = "your-dataset-id"       # @param {type:"string"}
BQ_LOCATION  = "US"                    # @param {type:"string"} e.g. "US", "EU", "us-central1"
GRAPH_NAME   = "your_property_graph"   # @param {type:"string"}

# ── Vertex AI Location ───────────────────────────────────────
VERTEXAI_LOCATION = "us-central1"      # @param {type:"string"}

# ── Domain Context ───────────────────────────────────────────
# Describe what your dataset is about in 2-3 sentences.
# This helps Gemini use sensible labels and relationship directions.
DOMAIN_CONTEXT = """
    Replace this with a short description of your dataset.
    For example: 'This is a retail dataset for an e-commerce company.
    It contains customers, their orders, products, and store locations.'
"""

# ── Business Questions ───────────────────────────────────────
# List the questions you want to answer with your graph.
# Include simple AND complex multi-hop questions for best results.
BUSINESS_QUESTIONS = [
    "Which customers have purchased the most products?",
    "What products are frequently bought together?",
    "Which customers visited more than 3 different stores?",
    "Which customers share similar purchasing behaviour?",
    "If a product is recalled, which customers need to be notified?",
    # Add your own questions here...
]

# ── Optional Hints ───────────────────────────────────────────
# List any tables you already know are junction/bridge tables.
# Leave empty to let Gemini decide.
KNOWN_EDGE_TABLES = []  # e.g. ["order_items", "product_tags"]

# Columns to exclude from the graph (embeddings, internal flags, etc.)
EXCLUDE_COLUMNS = []    # e.g. ["embedding", "internal_flag", "created_by"]

---
## Step 1: Extract Schema from BigQuery

We pull table schemas and a small sample of rows from `INFORMATION_SCHEMA`.
The sample rows help Gemini understand real data values, not just column names.

In [ ]:
def extract_schema(client: bigquery.Client) -> dict:
    """
    Extracts table/view schemas and sample rows from BigQuery INFORMATION_SCHEMA.
    Handles datasets in any region by passing BQ_LOCATION to all queries.
    Returns: dict of {table_name: {columns, row_count, type, sample_rows}}
    """
    job_config = bigquery.QueryJobConfig()
    location = BQ_LOCATION

    # Pull column-level schema — includes both TABLEs and VIEWs
    schema_df = client.query(f"""
        SELECT
            t.table_name,
            t.table_type,
            c.column_name,
            c.data_type,
            c.is_nullable,
            c.ordinal_position
        FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES` t
        JOIN `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS` c
            ON t.table_name = c.table_name
        WHERE t.table_type IN ('BASE TABLE', 'VIEW')
        ORDER BY t.table_name, c.ordinal_position
    """, job_config=bigquery.QueryJobConfig(default_dataset=f"{PROJECT_ID}.{DATASET_ID}"),
    location=location).to_dataframe()

    # Pull row counts (available for tables; views show 0)
    try:
        row_counts = {
            r["table_name"]: r["row_count"]
            for r in client.query(f"""
                SELECT table_name, row_count
                FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLE_STORAGE`
            """, location=location).to_dataframe().to_dict("records")
        }
    except Exception:
        row_counts = {}

    # Build schema dict with sample rows per table/view
    schema = {}
    for table in schema_df["table_name"].unique():
        table_rows = schema_df[schema_df["table_name"] == table]
        table_type = table_rows["table_type"].iloc[0]

        cols = table_rows[
            ["column_name", "data_type", "is_nullable"]
        ].to_dict("records")

        # Exclude specified columns
        if EXCLUDE_COLUMNS:
            cols = [c for c in cols if c["column_name"] not in EXCLUDE_COLUMNS]

        try:
            sample = client.query(
                f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{table}` LIMIT 3",
                location=location
            ).to_dataframe().to_dict("records")
        except Exception:
            sample = []

        schema[table] = {
            "type": table_type,
            "columns": cols,
            "row_count": row_counts.get(table, "view — see sample"),
            "sample_rows": sample,
        }

    return schema

In [ ]:
bq_client = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=VERTEXAI_LOCATION)
model = GenerativeModel("gemini-2.0-flash-001")

print("Extracting schema from BigQuery...")
print(f"  Project:  {PROJECT_ID}")
print(f"  Dataset:  {DATASET_ID}")
print(f"  Location: {BQ_LOCATION}\n")

schema = extract_schema(bq_client)

print(f"Found {len(schema)} tables/views:\n")
for table, info in schema.items():
    col_names = [c["column_name"] for c in info["columns"]]
    print(f"  {'📋' if info['type'] == 'BASE TABLE' else '👁️ '} {table} [{info['type']}] ({info['row_count']} rows)")
    print(f"     Columns: {', '.join(col_names)}")

---
## Step 1b: Filter Tables/Views Suitable for Graph

Not all tables and views in a dataset are good candidates for a property graph.
Pre-aggregated metrics, time-series summaries, and KPI dashboards are useful for analytics
but make poor nodes or edges.

This step asks Gemini to evaluate each table/view and rate its suitability,
so only entity-based and relationship-based objects are passed to the graph designer.

> **You can override Gemini's recommendations** by editing the `selected_tables` list before proceeding.

In [ ]:
def filter_graph_candidates(schema: dict, model: GenerativeModel) -> dict:
    """
    Asks Gemini to evaluate each table/view for graph suitability.
    Returns a dict with suitability ratings and reasoning for each table.
    """
    # Build a compact schema summary to avoid sending full sample data
    compact_schema = {
        name: {
            "type": info["type"],
            "columns": [c["column_name"] for c in info["columns"]],
            "sample": info["sample_rows"][:1]  # just 1 row for context
        }
        for name, info in schema.items()
    }

    prompt = f"""
You are a BigQuery Property Graph expert evaluating which tables and views from a dataset
are suitable for inclusion in a Property Graph.

DOMAIN CONTEXT:
{DOMAIN_CONTEXT}

BUSINESS QUESTIONS TO ANSWER:
{chr(10).join([f'  - {q}' for q in BUSINESS_QUESTIONS])}

TABLES AND VIEWS TO EVALUATE:
{json.dumps(compact_schema, indent=2, default=str)}

For each table/view, evaluate its suitability for a Property Graph using these criteria:

GOOD candidates (include):
- Entity tables: represent real-world objects with a clear identity (users, sessions, products, pages, channels)
- Relationship tables: connect two entities with foreign keys (junction tables, fact tables)
- Dimensional tables: describe attributes of an entity (device info, geography, campaign details)

BAD candidates (exclude):
- Pre-aggregated metrics: KPI summaries, daily rollups, bounce rates, growth metrics
- Time-series tables: data aggregated by day/week/month with no entity key
- Analytical views: designed for dashboards, not for traversal
- Duplicate/redundant views: if two views expose the same underlying entity, keep only the most complete one

Return ONLY valid JSON:
{{
  "evaluations": [
    {{
      "table": "table_name",
      "suitable": true,
      "role": "node | edge | either | exclude",
      "reasoning": "one sentence explanation",
      "suggested_label": "NodeLabel or EDGE_LABEL if suitable, null if not"
    }}
  ],
  "summary": "Brief summary of what was included and excluded and why"
}}
"""

    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(response_mime_type="application/json")
    )
    return json.loads(response.text)


def display_filtering_results(evaluation: dict) -> list:
    """Display the filtering results as a table and return list of selected table names."""
    rows = []
    for e in evaluation["evaluations"]:
        rows.append({
            "Table / View": e["table"],
            "Include?": "✅ Yes" if e["suitable"] else "❌ No",
            "Role": e.get("role", "-"),
            "Suggested Label": e.get("suggested_label") or "-",
            "Reasoning": e["reasoning"]
        })

    df = pd.DataFrame(rows)
    display(Markdown("### 🔍 Graph Suitability Evaluation"))
    display(df)
    display(Markdown(f"**Summary:** {evaluation['summary']}"))

    selected = [e["table"] for e in evaluation["evaluations"] if e["suitable"]]
    return selected

In [ ]:
print("Asking Gemini to evaluate graph suitability...\n")
evaluation = filter_graph_candidates(schema, model)
selected_tables = display_filtering_results(evaluation)

print(f"\n✅ {len(selected_tables)} tables/views selected for graph design:")
for t in selected_tables:
    print(f"   - {t}")
print(f"\n❌ {len(schema) - len(selected_tables)} excluded as unsuitable for graph.")

In [ ]:
# ── Override Gemini's selection if needed ───────────────────
# If Gemini excluded something you want to include (or vice versa),
# edit the list below before running Step 2.
#
# Example:
#   selected_tables = ["vw_users_base", "vw_sessions_base", "vw_content_performance"]

# selected_tables = selected_tables  # uncomment and edit to override

# Filter the schema to only include selected tables
filtered_schema = {k: v for k, v in schema.items() if k in selected_tables}
print(f"Proceeding with {len(filtered_schema)} tables/views for graph design.")

---
## Step 2: Identify Nodes & Edges with Gemini

Gemini analyzes your schemas and business questions to recommend:
- Which tables should be **nodes** (entities)
- Which tables should be **edges** (relationships)
- The direction and label of each edge
- Which properties to expose

> **Note:** A single table can serve as both a node AND an edge source.
> For example, an `orders` table can be an `Orders` node AND a `PLACED` edge from Customer to Order.

In [ ]:
def identify_graph_structure(schema: dict, model: GenerativeModel) -> dict:
    """
    Uses Gemini to identify nodes and edges from table schemas.
    Returns structured recommendations with reasoning.
    """
    questions_text = "\n".join([f"  - {q}" for q in BUSINESS_QUESTIONS])
    hints_text = (
        f"The following tables are known relationship/bridge tables: {KNOWN_EDGE_TABLES}"
        if KNOWN_EDGE_TABLES else ""
    )

    prompt = f"""
You are a BigQuery Property Graph expert. Analyze the following BigQuery table schemas
and design a Property Graph to best answer the given business questions.

DOMAIN CONTEXT:
{DOMAIN_CONTEXT}

BUSINESS QUESTIONS TO ANSWER:
{questions_text}

TABLE SCHEMAS (with sample data):
{json.dumps(schema, indent=2, default=str)}

{hints_text}

DESIGN RULES:
1. Node tables represent real-world entities with a clear primary key and descriptive attributes.
2. Edge tables represent relationships — they typically connect two entity tables via foreign keys.
3. A table CAN serve as both a node AND an edge source if it represents both an entity
   and a relationship (e.g. orders is an Orders node AND a PLACED edge from Customer to Order).
   In this case, include it once in nodes and once in edges with a different alias.
4. Edge direction should reflect the natural business flow (Customer PLACED Order, not reverse).
5. Use UPPER_SNAKE_CASE for edge labels (PLACED, HAS, VISITED, BOUGHT_TOGETHER).
6. Use PascalCase for node labels (Customer, Order, Product).
7. Only expose properties useful for filtering or querying. Exclude raw join keys where possible.
8. If a table is used as both a node and an edge, give the edge usage a descriptive alias.

Return ONLY valid JSON:
{{
  "nodes": [
    {{
      "table": "exact_table_name",
      "alias": "optional_alias_only_if_table_used_multiple_times",
      "label": "NodeLabel",
      "key_column": "primary_key_column",
      "properties": ["col1", "col2"],
      "reasoning": "why this is a node"
    }}
  ],
  "edges": [
    {{
      "table": "exact_table_name",
      "alias": "descriptive_alias",
      "label": "EDGE_LABEL",
      "key_column": "key_column_or_composite",
      "source_key": "foreign_key_column",
      "source_node_table": "references_this_table",
      "destination_key": "foreign_key_column",
      "destination_node_table": "references_this_table",
      "properties": ["col1"],
      "reasoning": "why this is an edge and explanation of direction"
    }}
  ],
  "graph_summary": "A paragraph summarising the graph structure and the questions it can answer."
}}
"""

    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(response_mime_type="application/json")
    )
    return json.loads(response.text)

In [ ]:
print("Asking Gemini to design your property graph...\n")
graph_structure = identify_graph_structure(filtered_schema, model)  # uses filtered schema

# Display nodes
display(Markdown("### 🔵 Recommended Nodes"))
nodes_data = []
for n in graph_structure["nodes"]:
    nodes_data.append({
        "Label": n["label"],
        "Table": n["table"],
        "Alias": n.get("alias", "-"),
        "Key": n["key_column"],
        "Properties": ", ".join(n.get("properties", ["ALL"])),
        "Reasoning": n["reasoning"]
    })
display(pd.DataFrame(nodes_data))

# Display edges
display(Markdown("### 🔴 Recommended Edges"))
edges_data = []
for e in graph_structure["edges"]:
    src_label = next(
        (n["label"] for n in graph_structure["nodes"] if n["table"] == e["source_node_table"]),
        e["source_node_table"]
    )
    dst_label = next(
        (n["label"] for n in graph_structure["nodes"] if n["table"] == e["destination_node_table"]),
        e["destination_node_table"]
    )
    edges_data.append({
        "Relationship": f"({src_label})-[{e['label']}]->({dst_label})",
        "Table": e["table"],
        "Alias": e.get("alias", "-"),
        "Key": e["key_column"],
        "Reasoning": e["reasoning"]
    })
display(pd.DataFrame(edges_data))

# Display summary
display(Markdown("### 📝 Graph Summary"))
display(Markdown(graph_structure["graph_summary"]))

---
## Step 3: Generate & Review the DDL

Review the generated `CREATE PROPERTY GRAPH` statement before creating it.
You can edit `graph_structure` above and re-run this cell if you want to make changes.

In [ ]:
def generate_ddl(graph_structure: dict) -> str:
    """Converts Gemini's recommendations into a CREATE PROPERTY GRAPH DDL statement."""

    def node_clause(n: dict) -> str:
        alias = f" AS {n['alias']}" if n.get("alias") else ""
        props = (
            f"PROPERTIES ({', '.join(n['properties'])})"
            if n.get("properties") else "PROPERTIES ALL COLUMNS"
        )
        return (
            f"  `{PROJECT_ID}.{DATASET_ID}.{n['table']}`{alias}\n"
            f"    KEY ({n['key_column']}) LABEL {n['label']} {props}"
        )

    def edge_clause(e: dict) -> str:
        alias = f" AS {e['alias']}" if e.get("alias") else ""
        props = (
            f"PROPERTIES ({', '.join(e['properties'])})"
            if e.get("properties") else "PROPERTIES ALL COLUMNS"
        )
        return (
            f"  `{PROJECT_ID}.{DATASET_ID}.{e['table']}`{alias}\n"
            f"    KEY ({e['key_column']})\n"
            f"    SOURCE KEY ({e['source_key']}) REFERENCES "
            f"`{PROJECT_ID}.{DATASET_ID}.{e['source_node_table']}`\n"
            f"    DESTINATION KEY ({e['destination_key']}) REFERENCES "
            f"`{PROJECT_ID}.{DATASET_ID}.{e['destination_node_table']}`\n"
            f"    LABEL {e['label']} {props}"
        )

    nodes_ddl = ",\n".join([node_clause(n) for n in graph_structure["nodes"]])
    edges_ddl = ",\n".join([edge_clause(e) for e in graph_structure["edges"]])

    return (
        f"CREATE OR REPLACE PROPERTY GRAPH "
        f"`{PROJECT_ID}.{DATASET_ID}.{GRAPH_NAME}`\n"
        f"NODE TABLES (\n{nodes_ddl}\n)\n"
        f"EDGE TABLES (\n{edges_ddl}\n);"
    )

In [ ]:
ddl = generate_ddl(graph_structure)

display(Markdown("### 📄 Generated DDL"))
print(ddl)

---
## Step 4: Create the Property Graph

Review the DDL above before running this cell.
Set `CREATE_GRAPH = True` when you are ready to create the graph.

In [ ]:
CREATE_GRAPH = False  # @param {type:"boolean"}

if CREATE_GRAPH:
    print(f"Creating property graph '{GRAPH_NAME}'...")
    bq_client.query(ddl).result()
    print(f"✅ Property graph '{GRAPH_NAME}' created successfully in {DATASET_ID}.")
else:
    print("Skipped. Set CREATE_GRAPH = True when you are ready to create the graph.")

---
## Step 5: Auto-generate Sample GQL Questions & Queries

Gemini generates 5 sample business questions with ready-to-run GQL queries,
tailored to your graph structure and the business questions you provided.

These are ordered from simple to complex — great as a starting point for an NL2GQL agent.

In [ ]:
def generate_sample_questions(graph_structure: dict, model: GenerativeModel) -> list:
    """
    Uses Gemini to generate sample NL questions + GQL queries for the property graph.
    Returns a list of {question, complexity, gql, why_gql} dicts.
    """
    prompt = f"""
You are a BigQuery GQL expert. Given the property graph structure below, generate
5 sample business questions ordered from simple to complex, each with a correct GQL query.

GRAPH STRUCTURE:
{json.dumps(graph_structure, indent=2)}

PROJECT:    {PROJECT_ID}
DATASET:    {DATASET_ID}
GRAPH NAME: {GRAPH_NAME}

BUSINESS QUESTIONS TO DRAW FROM:
{chr(10).join([f'  - {q}' for q in BUSINESS_QUESTIONS])}

GQL RULES — follow these exactly:
1. Always use this pattern:
   SELECT ... FROM GRAPH_TABLE(`{PROJECT_ID}.{DATASET_ID}.{GRAPH_NAME}` MATCH ... RETURN ...)
2. Aggregations (COUNT, SUM, STRING_AGG, etc.) must go in the outer SQL — NOT inside GRAPH_TABLE.
3. Return tabular results only — never use TO_JSON or path objects.
4. Always use explicit column aliases in the RETURN clause.
5. To filter on a node property use inline syntax: {{property_name: 'value'}}
6. For multi-hop queries use WITH to carry variables between MATCH clauses.
7. Use UPPER_SNAKE_CASE edge labels and PascalCase node labels exactly as defined in the graph.
8. Queries should return meaningful results, not just IDs.

Return ONLY a valid JSON array:
[
  {{
    "question": "The natural language question a business user would ask",
    "complexity": "simple | medium | complex",
    "hops": 1,
    "gql": "The complete, runnable SQL/GQL query",
    "why_gql": "One sentence on why GQL expresses this better than SQL"
  }}
]
"""

    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(response_mime_type="application/json")
    )
    return json.loads(response.text)

In [ ]:
print("Generating sample GQL questions...\n")
sample_questions = generate_sample_questions(graph_structure, model)

for i, q in enumerate(sample_questions, 1):
    display(Markdown(f"---\n### Q{i} [{q['complexity'].upper()} — {q['hops']}-hop] {q['question']}"))
    display(Markdown(f"**Why GQL:** {q['why_gql']}"))
    display(Markdown(f"```sql\n{q['gql']}\n```"))

---
## Step 6: Run Your First Query

Pick any of the generated queries above and run it here to verify your graph is working.
Replace the query below with whichever sample question interests you most.

In [ ]:
# Automatically runs the first sample question — replace with any query you like
first_query = sample_questions[0]["gql"]

print(f"Running: {sample_questions[0]['question']}\n")
print(f"Query:\n{first_query}\n")

result = bq_client.query(first_query).to_dataframe()
display(result)

---
## ✅ What's Next?

Now that your property graph is live, here are some things you can do:

| Next Step | How |
|---|---|
| Ask more questions | Use the sample queries as a starting point |
| Build an NL2GQL agent | Use the sample questions as golden query examples |
| Visualise the graph | Use `%%bigquery --graph` in a BigQuery notebook |
| Add more relationships | Edit `graph_structure` and re-run Step 3 & 4 |
| Mix in unstructured data | See the companion notebook: `bq_property_graph_builder_with_pdfs.ipynb` |

---
## Step 7: Generate NL2GQL Agent Instructions

Once your graph is built, you can use it as the backend for a conversational NL2GQL agent.

This step generates a complete set of agent instructions — covering graph structure, query rules, node/edge properties, filtering patterns, and reference data — ready to paste directly into your agent configuration.

In [ ]:
def generate_agent_instructions(
    graph_structure: dict,
    schema: dict,
    sample_questions: list,
    model: GenerativeModel
) -> str:
    """
    Uses Gemini to generate a complete set of NL2GQL agent instructions
    based on the graph structure, schema, and sample questions.
    """
    prompt = f"""
You are an expert at configuring NL2GQL agents that convert natural language questions
into BigQuery GQL queries.

Given the following property graph structure and sample queries, generate a complete set
of agent instructions that will help an LLM accurately convert natural language questions
into correct GQL queries for this specific graph.

DOMAIN CONTEXT:
{DOMAIN_CONTEXT}

GRAPH STRUCTURE:
{json.dumps(graph_structure, indent=2)}

FULL SCHEMA (for reference data):
{json.dumps(schema, indent=2, default=str)}

PROJECT:    {PROJECT_ID}
DATASET:    {DATASET_ID}
GRAPH NAME: {GRAPH_NAME}

Generate agent instructions covering ALL of the following sections:

1. **Graph & Query Structure** — how to always structure queries (GRAPH_TABLE pattern,
   where aggregations go, always use tabular output with aliases).

2. **Nodes & Properties** — for each node: what properties are exposed, which are
   safe to filter on, which are keys vs. queryable properties. Flag any node that
   only exposes limited properties (e.g. a Customer node that only exposes 'company').

3. **Edges & Traversal Patterns** — list each edge label, its direction, and when
   to use it. Include the full multi-hop pattern for impact analysis questions
   (e.g. if X changes, who is affected).

4. **Filtering Syntax** — how to filter on node properties using inline syntax,
   how to filter on edge properties, case sensitivity notes.

5. **Output Formatting** — what to always include in results for different question types.
   How to use STRING_AGG for returning multiple values per row.

6. **Reference Data** — list all unique values for key categorical fields extracted
   from the schema and sample data. Include all unique entity names, categories,
   types, and statuses visible in the data. This helps the agent use exact values
   rather than guessing.

7. **Common Mistakes to Avoid** — list 3-5 specific pitfalls based on this graph
   (e.g. using a property that isn't exposed, wrong edge direction, aggregating
   inside GRAPH_TABLE instead of the outer SQL).

Format the output as clean markdown with bold section headers.
Be specific and concrete — reference actual table names, column names, edge labels,
and property names from this graph. Do not give generic advice.
"""

    response = model.generate_content(prompt)
    return response.text

In [ ]:
print("Generating agent instructions...\n")
agent_instructions = generate_agent_instructions(
    graph_structure, schema, sample_questions, model
)

display(Markdown("## 🤖 NL2GQL Agent Instructions"))
display(Markdown(agent_instructions))

In [ ]:
# Save instructions + golden queries to a markdown file
# Ready to paste directly into your agent's system prompt or configuration
output_path = f"{GRAPH_NAME}_agent_instructions.md"

with open(output_path, "w") as f:
    f.write(f"# NL2GQL Agent Instructions\n")
    f.write(f"**Graph:** `{PROJECT_ID}.{DATASET_ID}.{GRAPH_NAME}`\n\n")
    f.write(agent_instructions)
    f.write("\n\n---\n## Golden Query Examples\n\n")
    for i, q in enumerate(sample_questions, 1):
        f.write(f"### Q{i} [{q['complexity'].upper()} — {q['hops']}-hop]\n")
        f.write(f"**Question:** {q['question']}\n\n")
        f.write(f"**Why GQL:** {q['why_gql']}\n\n")
        f.write(f"```sql\n{q['gql']}\n```\n\n")

print(f"✅ Saved to: {output_path}")
print("   Copy the contents of this file into your agent's system prompt or configuration.")